# tribe-bench: Setup & Smoke Test

Run this notebook on **Kaggle** (free T4 GPU) or **Google Colab**.

**Goal:** Verify that TRIBE v2 loads, runs inference on a short clip,
and produces output of the expected shape. Record VRAM usage.

**Time needed:** ~5 minutes.

## 1. Install Dependencies

In [ ]:
# Install tribe-bench from GitHub (update URL when repo is public)
# !pip install git+https://github.com/deveshb/tribe-bench.git

# For now, install from local clone
!pip install -e /kaggle/working/tribe-bench

# Install TRIBE v2 and its dependencies
# NOTE: neuralset/neuraltrain availability is UNVERIFIED (G016)
# If these fail, check https://github.com/facebookresearch/tribev2 for install instructions
!pip install tribev2
!pip install neuralset neuraltrain exca

## 2. Check GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Inference will not work.")

## 3. Verify tribe-bench Imports

In [ ]:
from tribe_tools import model, inference, atlas, viz, cache, video_utils
from brainlens import inference as bl_inference, attribution, visualization
from neurocheck.claims import load_claims, validate_claims

# Validate claims database
claims = load_claims()
errors = validate_claims()
print(f"Claims loaded: {len(claims)}, Errors: {len(errors)}")
print("tribe-bench imports: OK")

## 4. Load TRIBE v2 Model

In [ ]:
import time
from tribe_tools.model import load_model

# HuggingFace auth may be needed for LLaMA 3.2
# from huggingface_hub import login
# login(token="YOUR_HF_TOKEN")

start = time.time()
tribe_model = load_model(device="cuda")
load_time = time.time() - start
print(f"Model loaded in {load_time:.1f}s")

## 5. Run Single Prediction

In [ ]:
from pathlib import Path
from tribe_tools.model import predict_single

# TODO: Upload a short test video to Kaggle dataset or use a sample
video_path = Path("test_video.mp4")

torch.cuda.reset_peak_memory_stats()
start = time.time()

preds, segments = predict_single(tribe_model, video_path)

infer_time = time.time() - start
peak_vram = torch.cuda.max_memory_allocated() / 1e9

print(f"\n=== SMOKE TEST RESULTS ===")
print(f"Prediction shape: {preds.shape}")
print(f"Dtype: {preds.dtype}")
print(f"Value range: [{preds.min():.4f}, {preds.max():.4f}]")
print(f"Mean: {preds.mean():.4f}, Std: {preds.std():.4f}")
print(f"Segments kept: {len(segments)}")
print(f"Inference time: {infer_time:.1f}s")
print(f"Peak VRAM: {peak_vram:.2f} GB")
print(f"=========================")

## 6. Test Atlas Integration

In [ ]:
from tribe_tools.atlas import get_topk_rois, summarize_by_roi, list_regions

# Mean prediction across segments
mean_pred = preds.mean(axis=0)

# Top activated regions
top_regions = get_topk_rois(mean_pred, k=10)
print("Top 10 activated regions:")
for i, r in enumerate(top_regions, 1):
    print(f"  {i}. {r}")

# Total regions available
all_regions = list_regions()
print(f"\nTotal HCP-MMP1 regions: {len(all_regions)}")

## 7. Save Results

Save everything to persistent storage before session ends.

In [ ]:
import numpy as np

np.save("smoke_test_prediction.npy", preds)
print(f"Saved prediction: {preds.shape}")

# Record measurements for ops/source-of-truth.md
print(f"\n=== RECORD IN source-of-truth.md ===")
print(f"Output shape: {preds.shape}")
print(f"n_vertices: {preds.shape[1]}")
print(f"Peak VRAM: {peak_vram:.2f} GB")
print(f"Load time: {load_time:.1f}s")
print(f"Inference time: {infer_time:.1f}s")
print(f"Smoke test: PASSED")